# 🔥 Notebook 6: Advanced Cache Patterns

Production caching faces challenges that basic patterns don't address. Let's learn how to handle cache stampedes, hot keys, and versioning.

## Learning Objectives

By the end of this notebook, you'll understand:
- Cache stampede and how to prevent it
- Hot key problem and solutions
- Request coalescing
- Cache versioning for safe invalidation

---

🔍 **Open RedisInsight** at http://localhost:5540 to watch cache operations!

In [ ]:
import redis
import json
import time
import random
import threading
from concurrent.futures import ThreadPoolExecutor
from typing import Optional, Callable

r = redis.Redis(host='localhost', port=6379, decode_responses=True)

try:
    r.ping()
    print("✅ Redis connected")
except:
    print("❌ Redis not running. Start with: docker-compose up -d")

## ⚡ Cache Stampede Problem

In [ ]:
print("⚡ Cache Stampede Problem")
print("=" * 60)
print("""
WHAT HAPPENS:
─────────────────────────────────────────────────────────────

1. Popular cache key expires
   └─► All 10,000 concurrent requests see cache MISS

2. All 10,000 requests hit database simultaneously
   └─► Database gets crushed! 💥

3. Database slows down or crashes
   └─► Entire site goes down 😱

Timeline:
─────────────────────────────────────────────────────────────
  Time 0: Cache key exists, everyone happy ✅
  Time T: TTL expires, key deleted
  Time T+1ms: 10,000 requests all see MISS
  Time T+2ms: 10,000 DB queries start
  Time T+100ms: Database CPU at 100% 🔥
""")

In [ ]:
db_queries = 0
lock = threading.Lock()

def expensive_db_query(key: str) -> str:
    global db_queries
    with lock:
        db_queries += 1
    time.sleep(0.05)
    return f"data_for_{key}"

def naive_cache_get(key: str) -> str:
    cached = r.get(key)
    if cached:
        return cached
    
    value = expensive_db_query(key)
    r.setex(key, 60, value)
    return value

print("🔬 Simulating Cache Stampede")
print("=" * 60)

r.delete("stampede:test")
db_queries = 0

print("\n50 concurrent requests hitting expired cache key...")

with ThreadPoolExecutor(max_workers=50) as executor:
    futures = [executor.submit(naive_cache_get, "stampede:test") for _ in range(50)]
    results = [f.result() for f in futures]

print(f"\n📊 Results:")
print(f"   DB queries executed: {db_queries}")
print(f"   Expected (with proper handling): 1")
print(f"\n💥 {db_queries - 1} unnecessary DB queries!")

## 🔒 Solution 1: Distributed Lock

In [ ]:
def cache_get_with_lock(key: str) -> str:
    cached = r.get(key)
    if cached:
        return cached
    
    lock_key = f"lock:{key}"
    acquired = r.set(lock_key, "1", nx=True, ex=10)
    
    if acquired:
        try:
            value = expensive_db_query(key)
            r.setex(key, 60, value)
            return value
        finally:
            r.delete(lock_key)
    else:
        for _ in range(100):
            time.sleep(0.01)
            cached = r.get(key)
            if cached:
                return cached
        return expensive_db_query(key)

print("🔒 Solution: Distributed Lock")
print("=" * 60)

r.delete("stampede:locked_test")
r.delete("lock:stampede:locked_test")
db_queries = 0

print("\n50 concurrent requests with lock protection...")

with ThreadPoolExecutor(max_workers=50) as executor:
    futures = [executor.submit(cache_get_with_lock, "stampede:locked_test") for _ in range(50)]
    results = [f.result() for f in futures]

print(f"\n📊 Results:")
print(f"   DB queries executed: {db_queries}")
print(f"   ✅ Only ONE query hit the database!")

## 🎲 Solution 2: Probabilistic Early Refresh

In [ ]:
print("🎲 Probabilistic Early Refresh")
print("=" * 60)
print("""
Instead of all requests seeing expiration at once,
randomly refresh BEFORE TTL expires.

How it works:
─────────────────────────────────────────────────────────────
• Cache entry has TTL of 60 seconds
• At 50 seconds remaining: 1% chance of refresh
• At 30 seconds remaining: 5% chance of refresh  
• At 10 seconds remaining: 20% chance of refresh
• At 0 seconds: Everyone refreshes (but likely already done!)

Result: Refreshes spread out over time, no stampede!
""")

def cache_get_with_early_refresh(key: str, base_ttl: int = 60) -> str:
    cached = r.get(key)
    ttl = r.ttl(key)
    
    if cached and ttl > 0:
        time_until_expire = ttl
        refresh_probability = max(0, (base_ttl - time_until_expire) / base_ttl) * 0.1
        
        if random.random() < refresh_probability:
            value = expensive_db_query(key)
            r.setex(key, base_ttl, value)
            return value
        return cached
    
    value = expensive_db_query(key)
    r.setex(key, base_ttl, value)
    return value

## 🔥 Hot Key Problem

In [ ]:
print("🔥 Hot Key Problem")
print("=" * 60)
print("""
SCENARIO: Celebrity posts viral content
─────────────────────────────────────────────────────────────

• Taylor Swift posts on Instagram
• 10 million fans try to view it
• All requests go to ONE cache key: "post:12345"

PROBLEM:
─────────────────────────────────────────────────────────────
• Single Redis node handles ALL requests for that key
• Network bandwidth saturated
• Redis CPU at 100%
• Other keys on same node become slow

SOLUTION: Key Fanout
─────────────────────────────────────────────────────────────
• Store same data under multiple keys
• post:12345:0, post:12345:1, ... post:12345:9
• Clients randomly choose which suffix
• Load distributed across keys/servers
""")

In [ ]:
class HotKeyCache:
    def __init__(self, redis_client, replicas: int = 10):
        self.redis = redis_client
        self.replicas = replicas
        self.key_access_counts = {}
    
    def _is_hot_key(self, key: str) -> bool:
        count = self.key_access_counts.get(key, 0)
        return count > 100
    
    def _get_replica_key(self, key: str) -> str:
        if self._is_hot_key(key):
            suffix = random.randint(0, self.replicas - 1)
            return f"{key}:{suffix}"
        return key
    
    def get(self, key: str) -> Optional[str]:
        self.key_access_counts[key] = self.key_access_counts.get(key, 0) + 1
        replica_key = self._get_replica_key(key)
        return self.redis.get(replica_key)
    
    def set(self, key: str, value: str, ttl: int = 60):
        if self._is_hot_key(key):
            for i in range(self.replicas):
                self.redis.setex(f"{key}:{i}", ttl, value)
        else:
            self.redis.setex(key, ttl, value)

print("🔬 Hot Key Fanout Demo")
print("=" * 60)

hot_cache = HotKeyCache(r, replicas=5)

for i in range(150):
    hot_cache.get("viral:post:12345")

hot_cache.set("viral:post:12345", "Taylor Swift's viral post content")

print("\n📊 Keys created in Redis:")
keys = r.keys("viral:post:12345*")
for key in sorted(keys):
    print(f"   {key}")

print(f"\n✅ Load distributed across {len(keys)} keys!")
print("   👀 Check RedisInsight for 'viral:post:12345:*' keys!")

## 🔢 Cache Versioning

In [ ]:
print("🔢 Cache Versioning")
print("=" * 60)
print("""
PROBLEM: Cache invalidation race conditions
─────────────────────────────────────────────────────────────
1. Request A reads user from DB (name: "Alice")
2. User updates name to "Alicia" in DB
3. Cache invalidated
4. Request A writes stale "Alice" to cache
5. Everyone sees old name! 😱

SOLUTION: Version in cache key
─────────────────────────────────────────────────────────────
• Store version number in DB: users.version
• Cache key includes version: "user:1:v42"
• On update: increment version to v43
• Old cache (v42) becomes unreachable
• No explicit invalidation needed!
""")

In [ ]:
class VersionedCache:
    def __init__(self, redis_client):
        self.redis = redis_client
        self.versions = {}
    
    def _versioned_key(self, key: str) -> str:
        version = self.versions.get(key, 1)
        return f"{key}:v{version}"
    
    def get(self, key: str) -> Optional[str]:
        versioned_key = self._versioned_key(key)
        return self.redis.get(versioned_key)
    
    def set(self, key: str, value: str, ttl: int = 300):
        versioned_key = self._versioned_key(key)
        self.redis.setex(versioned_key, ttl, value)
    
    def increment_version(self, key: str):
        self.versions[key] = self.versions.get(key, 1) + 1
        return self.versions[key]

print("🔬 Cache Versioning Demo")
print("=" * 60)

vcache = VersionedCache(r)

print("\n1. Cache user with version 1:")
vcache.set("user:1", json.dumps({"name": "Alice"}))
print(f"   Key: user:1:v1")
print(f"   Value: {vcache.get('user:1')}")

print("\n2. User updates name, increment version:")
new_version = vcache.increment_version("user:1")
vcache.set("user:1", json.dumps({"name": "Alicia"}))
print(f"   New version: {new_version}")
print(f"   Key: user:1:v{new_version}")
print(f"   Value: {vcache.get('user:1')}")

print("\n3. Old cache key still exists but unreachable:")
old_value = r.get("user:1:v1")
print(f"   user:1:v1 = {old_value}")
print(f"   (Will expire via TTL, no race condition!)")

print("\n💡 No explicit invalidation - old keys just become orphans!")

## 🧪 Quick Quiz

1. **What causes a cache stampede?**

2. **How does key fanout solve the hot key problem?**

3. **Why does cache versioning avoid race conditions?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Cache stampede cause:")
print("   - Popular key expires")
print("   - All requests see cache miss simultaneously")
print("   - All hit database at once")
print()
print("2. Key fanout for hot keys:")
print("   - Store same data under multiple keys")
print("   - Clients randomly select which key")
print("   - Load distributed across keys/servers")
print()
print("3. Cache versioning avoids races:")
print("   - Version is part of cache key")
print("   - Update increments version")
print("   - Old cache becomes unreachable")
print("   - Late writers can't overwrite new data")

## 📚 Summary

### Key Takeaways

1. **Cache stampede** - Use locks or early refresh to prevent DB overload
2. **Hot keys** - Fanout across multiple keys for viral content
3. **Request coalescing** - Combine duplicate in-flight requests
4. **Cache versioning** - Avoids invalidation race conditions
5. **Monitor hit rates** - Low hit rate indicates problems

### Interview Tips

> "For hot keys like celebrity posts, I'd use key fanout - storing the same data under multiple keys like `post:123:0` through `post:123:9`. Clients randomly choose which suffix, distributing load across cache nodes."

> "To prevent cache stampedes, I'd use a distributed lock so only one request rebuilds the cache. Other requests wait briefly for the first one to complete. For critical paths, I'd also use probabilistic early refresh."

### The Complete Read Scaling Toolkit

```
┌─────────────────────────────────────────────────────────────┐
│  1. Indexing          → 10x-100x improvement               │
│  2. Denormalization   → Eliminate joins                    │
│  3. Read replicas     → Scale horizontally                 │
│  4. Application cache → Sub-millisecond reads              │
│  5. CDN               → Global edge caching                │
└─────────────────────────────────────────────────────────────┘
```